<img src=https://audiovisuales.icesi.edu.co/assets/custom/images/ICESI_logo_prin_descriptor_RGB_POSITIVO_0924.jpg width=200>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juanestebancg2806/sugarcane-yield-quality-prediction/blob/main/03_comparacion_modelos_ipsa.ipynb)

# **Proyecto Caña de Azúcar**
## Modelado y generación de insights para el TCH y la sacarosa en el Ingenio Providencia


--------------------------------------------------------------------

**Aprendizaje Automático II - Proyecto final**

**Profesor:** Anibal Sosa

**Integrantes:**

*   Juan David Martínez Legarda
*   Daniel Velasco López
*   Juan Esteban Cardona García
*   Katherin Adriana Camargo Cetina

Fecha: 22/11/2026

________________________________________________________________________

# IPSA (CC01-1940): comparación de modelos de clasificación

En este notebook se construyen y comparan modelos para clasificar cada cosecha de la variedad CC01-1940 en un nivel bajo, medio o alto de tonelaje (`TCH`, toneladas de caña por hectárea) y de sacarosa (porcentaje de sacarosa en la caña). Los dos targets se tratan como problemas independientes.

**Escenario de uso.** Los modelos clasifican una cosecha en el momento de programar su corte. En ese momento se conocen el número de corte, la dosis de madurante y el grupo de tenencia, y, como la fecha de corte ya está programada, también la edad del cultivo, las semanas desde la aplicación del madurante y el mes de cosecha. La lluvia registrada se acumula hasta la cosecha con una ventana que no está documentada, así que solo se conoce completa cerca del corte. Por eso los modelos describen la programación del corte y no un pronóstico con meses de anticipación. Las variables que se miden en la cosecha o en el molino quedan fuera de los predictores.

El recorrido va de lo simple a lo complejo: dos clasificadores sin predictores como piso, la regresión logística multinomial con regularización y KNN como modelos de referencia, y Random Forest, XGBoost y máquinas de vectores de soporte (SVM) como modelos avanzados. Después se prueban dos alternativas: agregar el resultado de la cosecha anterior de cada suerte, y predecir el valor continuo de cada target con un modelo de regresión para asignar la clase con los umbrales del Ingenio.

Para que la comparación sea justa, todos los modelos comparten el preprocesamiento, la partición en entrenamiento y prueba, los pliegues de validación cruzada y el criterio de selección. Así, una diferencia de desempeño se puede atribuir al modelo y no a la forma de evaluarlo.

**Criterios generales**

- La partición 80/20 se hace al inicio y se agrupa por periodo de cosecha: ningún mes de cosecha aparece a la vez en entrenamiento y en prueba.
- Los hiperparámetros y la selección de modelos se deciden con validación cruzada sobre el entrenamiento, con la misma agrupación: 5 pliegues repetidos con 3 asignaciones distintas de los periodos, para un total de 15 pliegues fijos y comunes a todos los modelos.
- La imputación, la codificación y el escalado se estiman dentro de un `Pipeline`, de modo que en cada pliegue solo usan los datos de entrenamiento de ese pliegue.
- La regla para elegir el mejor modelo se fija antes de comparar los modelos.
- El conjunto de prueba se usa una sola vez, al final, con todos los modelos ya elegidos.

**Estructura**

1. Carga del dataset procesado
2. Preprocesamiento
3. Partición, pliegues de validación cruzada y auditoría de fuga de información
4. Métricas y regla de elección del modelo
5. Modelos de referencia: clasificadores sin predictores, regresión logística multinomial y KNN
6. Modelos avanzados: Random Forest, XGBoost, SVM, historial de la suerte y regresión con umbrales
7. Comparación en validación cruzada y elección del modelo
8. Evaluación en el conjunto de prueba
9. Interpretabilidad
10. Conclusiones

## Punto de partida: qué dejó el análisis exploratorio

El EDA y la preparación de los datos se hicieron en `01_eda_ipsa.ipynb`. A continuación se resume lo que se necesita de ese trabajo para el modelado.

**Los datos.** `BD_IPSA_1940.xlsx` es un subconjunto del Ingenio Providencia con 2187 cosechas entre julio de 2014 y enero de 2021. Todas comparten tres condiciones: la variedad CC01-1940, la maduración química con BONUS 250 EC y la cosecha mecanizada en verde. Las conclusiones aplican solo a suertes con esas características. Cada fila es una suerte (identificada por la hacienda `FAZ` y la suerte `TAL`) cosechada en un mes (`periodo`, en formato AAAAMM). El dataset tiene 1115 suertes de 285 haciendas, y 629 de esas suertes aparecen en dos o más cosechas, así que los registros no son independientes entre sí.

**Las clases.** Cada target se divide en tres niveles con los umbrales del Ingenio:

| Clase | `TCH` (t/ha) | Sacarosa (%) |
|---|---|---|
| Bajo | menos de 125 | menos de 12.2 |
| Medio | de 125 a 150 | de 12.2 a 12.9 |
| Alto | más de 150 | desde 13.0 |

Los umbrales son valores fijos que no dependen de los datos, así que construir las clases no filtra información del conjunto de prueba. El umbral inferior coincide con el percentil 25 de cada target, y el superior queda cerca del percentil 60. La clase baja, que agrupa las cosechas que el Ingenio necesita detectar para intervenir, es la más pequeña. Los 4 registros con `TCH` menor que 60 t/ha se consideraron errores de registro: no tienen clase de `TCH`, pero conservan la de sacarosa. Los dos targets tienen una correlación débil (Spearman de −0.21), así que se modelan como dos problemas independientes.

**Los predictores y su justificación.** El modelo principal usa 7 variables originales, comunes a los dos targets:

| Predictor | Qué mide | Por qué entra | Tratamiento |
|---|---|---|---|
| `edad` | Edad del cultivo en la cosecha (meses) | Más edad, más biomasa acumulada; en sacarosa la relación tiene forma de U invertida | Sin nulos |
| `cortes` | Número de corte de la cepa | El tonelaje baja con cada corte sucesivo | Sin nulos |
| `semsmad` | Semanas entre la aplicación del madurante y el corte | La sacarosa sube hasta unas 12 semanas y luego se estabiliza | 2 valores negativos tratados como faltantes |
| `dosismad` | Dosis de madurante (L/ha) | Fundamento agronómico. En el EDA no mostró relación con los targets, pero el 98 % de las dosis está entre 0.5 y 1.5 L/ha, un rango demasiado estrecho para descartar su efecto | 2 valores inválidos (0.0 y 9.0) tratados como faltantes |
| `lluvias` | Lluvia registrada (mm), con ventana de acumulación no documentada | Principal predictor numérico de la sacarosa (Spearman de −0.28) | El 31.3 % son ceros que corresponden a falta de medición; se imputan con la mediana del mes de cosecha y se acotan en el percentil 99 |
| `mes` | Mes de cosecha | La sacarosa tiene dos máximos al año, tras cada temporada seca del valle del río Cauca [2] | Dos armónicos (periodos de 12 y 6 meses), que capturan el 93 % de la variación mensual de la sacarosa |
| `grupo_tenencia` | Grupo de tenencia de la hacienda (códigos 1, 2 y 3 sin significado documentado) | Relación débil con `TCH` | Variables indicadoras, con el código 1 como referencia |

Con los cuatro armónicos del mes y las dos indicadoras de la tenencia, las 7 variables generan 11 columnas en la matriz de diseño. En el EDA, ningún predictor numérico candidato tuvo un VIF mayor que 5. El VIF de la matriz de diseño completa, que incluye los armónicos y las indicadoras, se calcula con el conjunto de entrenamiento en la sección 3.

**Lo que quedó fuera de los predictores**

| Columnas | Motivo |
|---|---|
| `NOME`, `FAZ`, `TAL` | Identifican la cosecha, no describen el cultivo. `FAZ` y `TAL` se usan para identificar cada suerte |
| `variedad`, `madurada`, `producto`, `tipocorte` | Son constantes en todo el dataset |
| `vejez` (horas entre el corte y el molino) y `me` (materia extraña, con confianza moderada) | Se miden al recibir la caña en el molino, después del corte. Usarlas sería fuga de información |
| `pct_diatrea` (porcentaje de entrenudos barrenados por *Diatraea*) | Según la metodología de Cenicaña, se mide en la cosecha, muestreando 100 tallos por suerte y contando los entrenudos barrenados [1]. Los datos no muestran que el Ingenio la estime antes del corte, así que no está disponible en el escenario de uso. Su aporte se evalúa como prueba de robustez en la sección 7 |
| `TCH` y sacarosa como predictores del otro target | Los dos se conocen solo después de la cosecha |
| `periodo` | Se usa para agrupar la partición y la validación cruzada, no como predictor. Su parte estacional entra por `mes` |
| `anio` | En el EDA, su relación con `TCH` la dominaba un solo año (2017, el de mayor lluvia), y un modelo aplicado a cosechas futuras no puede extrapolar el efecto de años que no vio. En el análisis de regresión del histórico de suertes, además, el efecto del año reflejaba condiciones propias de cada periodo, comunes a todas las fincas, que no se trasladan a una zafra nueva. Su aporte se evalúa como prueba de robustez en la sección 7 |
| `lluvia_sin_dato` | En el EDA, los registros sin lluvia medida no difieren de los demás en ninguno de los dos targets: su información mutua con ambos quedó por debajo del nivel de ruido |

**Relaciones que encontró el EDA**

- Todas las relaciones con los targets son débiles: ningún predictor numérico supera una correlación de Spearman de 0.29 en valor absoluto. Se espera un poder predictivo limitado.
- El tonelaje se relaciona sobre todo con el número de corte (Spearman de −0.27) y con la edad del cultivo (0.15), y no tiene estacionalidad: sus medianas mensuales van de 141 a 152 t/ha.
- La sacarosa depende sobre todo de la época de cosecha: el mes explica el 21 % de su variación, con medianas más altas en marzo, septiembre y octubre (13.2 % a 13.4 %) y más bajas entre abril y julio (12.1 % a 12.5 %). Más lluvia se asocia con menos sacarosa.
- Dos relaciones con la sacarosa no son lineales: con la edad tiene forma de U invertida (sube de 12.5 % a cerca de 13.0 % hacia los 12.5 meses y baja después), y con las semanas de madurante crece hasta unas 12 semanas y luego se estabiliza. Un modelo lineal en esas variables no las captura por completo.

**Limitaciones de los datos**

- `lluvias` es un único valor por registro, con una ventana de acumulación no documentada, y casi un tercio de sus valores no tiene medición.
- El significado de `me` y de los códigos de `grupo_tenencia` no está confirmado por el Ingenio, y el de `semsmad` se dedujo de los datos.
- Parte de la variación de la sacarosa depende de factores posteriores al corte (materia extraña y horas hasta el molino), que no están disponibles antes de la cosecha.

In [19]:
# Librerías generales del notebook. Los módulos de scikit-learn y xgboost se importan en la sección donde se usan.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import xgboost

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

%matplotlib inline

# Versiones usadas, para la reproducibilidad de los resultados
print(f"pandas {pd.__version__} | numpy {np.__version__} | scikit-learn {sklearn.__version__} | "
      f"xgboost {xgboost.__version__}")

pandas 3.0.6 | numpy 2.5.4 | scikit-learn 1.9.1 | xgboost 3.4.1


## 1. Carga del dataset procesado

La última celda de `01_eda_ipsa.ipynb` guarda el dataset en `datos/ipsa_limpio.parquet`. El archivo trae solo transformaciones que dan el mismo resultado en cada fila sin importar las demás: los valores inválidos de `lluvias`, `dosismad` y `semsmad` convertidos en faltantes, la marca `excluir_tch` y las clases con los umbrales del Ingenio. No trae ninguna imputación, porque los estadísticos para imputar deben calcularse solo con el conjunto de entrenamiento.

Antes de modelar se verifica que el archivo conserve lo establecido en el EDA:

- 2187 cosechas y 22 columnas, sin repetir la combinación de hacienda, suerte y periodo.
- Los faltantes de los predictores: 685 en `lluvias`, 2 en `semsmad` y 2 en `dosismad`. Si alguno cambia, el archivo llegó imputado.
- 4 registros con `excluir_tch`, sin clase de `TCH`, y clase de sacarosa en todos los registros.
- Que cada clase se pueda reconstruir con los umbrales del Ingenio a partir de su propio target. Así se confirma que las clases no dependen de ningún estadístico del dataset.

Además, las clases se codifican como enteros ordenados (bajo = 0, medio = 1, alto = 2), porque XGBoost no acepta etiquetas de texto. Es un mapeo fijo, sin estadísticos. Los armónicos del mes que trae el archivo no se usan: se recalculan dentro del pipeline a partir de `mes` (sección 2). `pct_diatrea` y `anio` se conservan en el DataFrame solo para las pruebas de robustez de la sección 7. Si alguna verificación falla, el notebook se detiene.

In [20]:
# 1. Carga del dataset procesado (salida de 01_eda_ipsa.ipynb) y verificación de sus propiedades
RANDOM_STATE = 42  # semilla común a la partición, los pliegues y los modelos
RUTA_IPSA = Path("datos/ipsa_limpio.parquet")

# Propiedades del archivo establecidas en 01_eda_ipsa.ipynb
FILAS_ESPERADAS, COLUMNAS_ESPERADAS = 2187, 22
NULOS_ESPERADOS = {"lluvias": 685, "semsmad": 2, "dosismad": 2}
FILAS_EXCLUIR_TCH = 4

# Predictores del modelo principal en su forma original. mes entra como número: sus armónicos se calculan en el pipeline
PREDICTORES_NUMERICOS = ["edad", "cortes", "semsmad", "lluvias", "dosismad"]
PREDICTOR_MES = "mes"
PREDICTORES_CATEGORICOS = ["grupo_tenencia"]
PREDICTORES = PREDICTORES_NUMERICOS + [PREDICTOR_MES] + PREDICTORES_CATEGORICOS

# Variables excluidas del modelo principal cuyo aporte se evalúa como robustez en la sección 7
PREDICTORES_ROBUSTEZ = ["pct_diatrea", "anio"]

# Columnas del archivo que no entran a ningún modelo (lluvia_sin_dato) o que se recalculan en el pipeline (armónicos)
COLUMNAS_NO_USADAS = ["lluvia_sin_dato", "mes_sin_12", "mes_cos_12", "mes_sin_6", "mes_cos_6"]

# Clases ordenadas, columna de clase de cada target y umbrales del Ingenio
CLASES = ["bajo", "medio", "alto"]
COLUMNA_CLASE = {"TCH": "clase_tch_ingenio", "sacarosa": "clase_sac_ingenio"}
COLUMNA_CODIGO = {"TCH": "codigo_tch", "sacarosa": "codigo_sac"}  # clase como entero: bajo 0, medio 1, alto 2
UMBRALES_INGENIO = {"TCH": (125, 150), "sacarosa": (12.2, 13.0)}


def formato_miles(numero):
    """Entero con espacio como separador de miles desde cinco cifras (norma RAE)."""
    return f"{numero:,}".replace(",", " ") if abs(numero) >= 10_000 else str(numero)


df_ipsa = pd.read_parquet(RUTA_IPSA)

# Estructura del archivo
assert df_ipsa.shape == (FILAS_ESPERADAS, COLUMNAS_ESPERADAS), f"Dimensiones inesperadas: {df_ipsa.shape}"
columnas_requeridas = PREDICTORES + PREDICTORES_ROBUSTEZ + COLUMNAS_NO_USADAS + list(COLUMNA_CLASE.values())
faltan = set(columnas_requeridas) - set(df_ipsa.columns)
assert not faltan, f"Faltan columnas en el archivo: {faltan}"
assert not df_ipsa.duplicated(subset=["FAZ", "TAL", "periodo"]).any(), "Hay cosechas repetidas"

# Faltantes originales: si cambian, el archivo llegó imputado
nulos = df_ipsa[PREDICTORES + PREDICTORES_ROBUSTEZ].isna().sum()
assert nulos[nulos > 0].to_dict() == NULOS_ESPERADOS, f"Faltantes inesperados: {nulos[nulos > 0].to_dict()}"

# Registros excluidos de TCH y completitud de las clases
assert df_ipsa["excluir_tch"].sum() == FILAS_EXCLUIR_TCH, "El número de registros con excluir_tch no es 4"
assert df_ipsa.loc[df_ipsa["excluir_tch"], "clase_tch_ingenio"].isna().all(), "Registros excluidos con clase de TCH"
assert df_ipsa.loc[~df_ipsa["excluir_tch"], "clase_tch_ingenio"].notna().all(), "Registros válidos sin clase de TCH"
assert df_ipsa["clase_sac_ingenio"].notna().all(), "Registros sin clase de sacarosa"
for columna in COLUMNA_CLASE.values():
    assert list(df_ipsa[columna].cat.categories) == CLASES and df_ipsa[columna].cat.ordered, \
        f"{columna} perdió el orden bajo < medio < alto"

# Las clases se reconstruyen con los umbrales del Ingenio a partir de su propio target
# TCH se registra en enteros: bajo < 125, medio de 125 a 150, alto > 150
assert pd.api.types.is_integer_dtype(df_ipsa["TCH"]), "TCH ya no está en valores enteros"
validos_tch = ~df_ipsa["excluir_tch"]
bajo_tch, alto_tch = UMBRALES_INGENIO["TCH"]
clase_tch_reconstruida = pd.cut(df_ipsa.loc[validos_tch, "TCH"], bins=[-np.inf, bajo_tch - 1, alto_tch, np.inf],
                                labels=CLASES)
# Sacarosa se registra con un decimal: bajo < 12.2, medio de 12.2 a 12.9, alto >= 13.0
bajo_sac, alto_sac = UMBRALES_INGENIO["sacarosa"]
clase_sac_reconstruida = pd.cut(df_ipsa["sacarosa"], bins=[-np.inf, bajo_sac, alto_sac, np.inf],
                                labels=CLASES, right=False)
assert (clase_tch_reconstruida.astype(str) == df_ipsa.loc[validos_tch, "clase_tch_ingenio"].astype(str)).all(), \
    "La clase de TCH no corresponde a los umbrales del Ingenio"
assert (clase_sac_reconstruida.astype(str) == df_ipsa["clase_sac_ingenio"].astype(str)).all(), \
    "La clase de sacarosa no corresponde a los umbrales del Ingenio"

# Codificación entera de las clases (mapeo fijo, sin estadísticos). Sin clase -> faltante
for target, columna in COLUMNA_CLASE.items():
    df_ipsa[COLUMNA_CODIGO[target]] = (df_ipsa[columna].cat.codes.astype("Int64")
                                       .mask(df_ipsa[columna].isna()))

# Identificador de suerte (hacienda + suerte), necesario para el historial y la auditoría de fuga
df_ipsa["suerte"] = df_ipsa["FAZ"].astype(str) + "_" + df_ipsa["TAL"]

print(f"Cosechas: {len(df_ipsa)} | haciendas: {df_ipsa['FAZ'].nunique()} | suertes: {df_ipsa['suerte'].nunique()} | "
      f"periodos: {df_ipsa['periodo'].nunique()} (de {df_ipsa['periodo'].min()} a {df_ipsa['periodo'].max()})")
print(f"Predictores del modelo principal: {len(PREDICTORES)} variables originales ({', '.join(PREDICTORES)})")

# Balance de clases de cada target: registros (enteros) y porcentaje
conteos = pd.DataFrame({target: df_ipsa[columna].value_counts().reindex(CLASES)
                        for target, columna in COLUMNA_CLASE.items()})
conteos.loc["total"] = conteos.sum()
porcentajes = (conteos / conteos.loc["total"] * 100).round(1)
display(pd.concat({"registros": conteos, "%": porcentajes}, axis=1))

Cosechas: 2187 | haciendas: 285 | suertes: 1115 | periodos: 72 (de 201407 a 202101)
Predictores del modelo principal: 7 variables originales (edad, cortes, semsmad, lluvias, dosismad, mes, grupo_tenencia)


registros               %         
            TCH sacarosa    TCH sacarosa
bajo        494      471   22.6     21.5
medio       850      802   38.9     36.7
alto        839      914   38.4     41.8
total      2183     2187  100.0    100.0

### Conclusión de la sección 1

El archivo cumple todas las verificaciones: tiene 2187 cosechas y 22 columnas, sin cosechas repetidas, los faltantes de los predictores son los originales del EDA y las clases de los dos targets se reconstruyen exactamente con los umbrales del Ingenio a partir de su propio valor. La imputación queda por hacer dentro del modelado, y las clases no dependen de ningún estadístico del dataset.

Las cosechas provienen de 285 haciendas y 1115 suertes, en 72 periodos entre julio de 2014 y enero de 2021. Muchas suertes aparecen en varios periodos, y la partición tiene que tener en cuenta esa estructura (sección 3).

El modelo principal usa 7 variables originales. Las clases quedan con este balance:

| Clase | `TCH` | Sacarosa |
|---|---|---|
| Bajo | 494 (22.6 %) | 471 (21.5 %) |
| Medio | 850 (38.9 %) | 802 (36.7 %) |
| Alto | 839 (38.4 %) | 914 (41.8 %) |
| Total | 2183 | 2187 |

En los dos targets la clase baja es la más pequeña, con poco más de la mitad de registros que cada una de las otras dos. El desbalance es moderado, pero suficiente para que un modelo que nunca predice la clase baja obtenga una accuracy aceptable. Por eso la métrica que decide se fija en la sección 4, y la ponderación de clases se trata como un hiperparámetro.

## 2. Preprocesamiento

El preprocesamiento aplica las reglas de imputación y codificación definidas en el EDA. Se implementa con transformadores de scikit-learn dentro de un `Pipeline`, y no como pasos sueltos sobre el DataFrame, por dos razones:

- En la validación cruzada y en la búsqueda de hiperparámetros, el `Pipeline` se reajusta en cada pliegue, de modo que cada estadístico se calcula solo con los datos de ajuste de ese pliegue.
- El conjunto de prueba recibe exactamente las mismas transformaciones, con los valores aprendidos en el entrenamiento.

### 2.1 Reglas y transformadores

Las reglas son de dos tipos. Las **sin estado** dan el mismo resultado en cada fila sin importar las demás, así que se aplican igual en entrenamiento y en prueba. Las **estimadas en el ajuste** calculan un estadístico con los datos de ajuste.

| Columnas | Transformación | Tipo |
|---|---|---|
| `lluvias` (con `mes` como apoyo) | Los faltantes se reemplazan con la mediana de `lluvias` de su mes de cosecha, calculada con los valores observados. Si un mes no tiene valores observados en el ajuste, se usa la mediana general. Después, los valores se acotan en el percentil 99 de los observados, para que unos pocos registros extremos no dominen las distancias de KNN y SVM | Estimada |
| `edad`, `cortes`, `semsmad`, `dosismad` | Faltantes reemplazados con la mediana. Solo `semsmad` y `dosismad` tienen faltantes (2 cada una) | Estimada |
| `mes` | Seno y coseno con periodos de 12 y 6 meses: $\sin(2\pi m/12)$, $\cos(2\pi m/12)$, $\sin(2\pi m/6)$ y $\cos(2\pi m/6)$. Diciembre y enero quedan contiguos, y el segundo par permite representar los dos máximos anuales de la sacarosa | Sin estado |
| `grupo_tenencia` | Indicadoras de los códigos 2 y 3, con el código 1 como referencia. Las categorías se fijan de antemano, así que todos los pliegues producen las mismas columnas | Sin estado |
| Todas las columnas | Estandarización con la media y la desviación del ajuste | Estimada |

Cuatro decisiones adicionales:

- **`mes` no entra como número.** Solo llega al modelo a través de sus armónicos, y la imputación de `lluvias` lo usa para elegir la mediana. Los armónicos se calculan dentro del `Pipeline` a partir de `mes`, y no se toman del archivo del EDA, para que la importancia por permutación de la sección 9 mueva el mes completo, con sus cuatro columnas a la vez.
- **El escalado es opcional.** La regresión logística lo necesita para que la penalización trate por igual a todos los coeficientes, y KNN y SVM porque miden distancias. Los árboles de Random Forest y XGBoost dividen cada variable por umbrales, que no cambian con una transformación monótona, así que se ajustan sin escalar.
- **Las indicadoras también se escalan** cuando hay escalado, para que pesen igual que las variables continuas en las distancias de KNN y SVM y en la penalización de la logística.
- **El preprocesador admite variables numéricas adicionales,** que se imputan con la mediana. Se usa en las pruebas de robustez con `pct_diatrea` y `anio` (sección 7). El historial de la suerte tiene su propio transformador, que se define en la sección 6.

El preprocesador devuelve un `DataFrame` con los nombres de las columnas, que se necesitan para leer los coeficientes y las importancias en la sección 9.

Antes de la partición no existe un conjunto de entrenamiento, y ajustar el preprocesador con todos los datos usaría el conjunto de prueba. Por eso los transformadores se verifican aquí con un ejemplo pequeño construido a mano, cuyos resultados se pueden calcular sin el código. Las verificaciones sobre los datos reales (que no queden faltantes, el escalado y el VIF) se hacen con el conjunto de entrenamiento, en la sección 3. La única comprobación que usa todo el dataset es la de los armónicos, porque es una transformación sin estado: se comparan con los que calculó el EDA.

In [21]:
# 2.1 Transformadores propios: lluvias por mes de cosecha y armónicos del mes
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted


class ImputadorLluvias(BaseEstimator, TransformerMixin):
    """Imputa lluvias con la mediana de su mes de cosecha y acota los valores al percentil indicado.

    Recibe dos columnas en este orden: lluvias y mes. Devuelve solo lluvias.
    Las medianas y el tope se calculan con los valores observados del conjunto de ajuste.
    """

    def __init__(self, percentil_tope=99):
        self.percentil_tope = percentil_tope

    def fit(self, X, y=None):
        lluvias, mes = self._separar(X)
        observado = ~np.isnan(lluvias)
        self.mediana_mes_ = pd.Series(lluvias[observado]).groupby(mes[observado]).median().to_dict()
        self.mediana_global_ = float(np.median(lluvias[observado]))
        self.tope_ = float(np.percentile(lluvias[observado], self.percentil_tope))
        self.n_features_in_ = 2
        return self

    def transform(self, X):
        check_is_fitted(self, "tope_")
        lluvias, mes = self._separar(X)
        # Mediana del mes; si el mes no tuvo datos observados en el ajuste, mediana general
        mediana = pd.Series(mes).map(self.mediana_mes_).fillna(self.mediana_global_).to_numpy()
        lluvias = np.where(np.isnan(lluvias), mediana, lluvias)
        return np.minimum(lluvias, self.tope_).reshape(-1, 1)

    def get_feature_names_out(self, input_features=None):
        return np.array(["lluvias"], dtype=object)

    @staticmethod
    def _separar(X):
        X = np.asarray(X, dtype=float)
        return X[:, 0], X[:, 1].astype(int)


class ArmonicosMes(BaseEstimator, TransformerMixin):
    """Seno y coseno del mes de cosecha con periodos de 12 y 6 meses. Transformación fija por fila: no aprende
    ningún estadístico, así que da el mismo resultado en cualquier pliegue."""

    PERIODOS = (12, 6)

    def fit(self, X, y=None):
        self.n_features_in_ = 1
        return self

    def transform(self, X):
        mes = np.asarray(X, dtype=float).reshape(-1)
        if not np.isin(mes, np.arange(1, 13)).all():
            raise ValueError("El mes de cosecha debe ser un entero entre 1 y 12")
        columnas = []
        for periodo in self.PERIODOS:
            angulo = 2 * np.pi * mes / periodo
            columnas += [np.sin(angulo), np.cos(angulo)]
        return np.column_stack(columnas)

    def get_feature_names_out(self, input_features=None):
        return np.array([f"mes_{funcion}_{periodo}" for periodo in self.PERIODOS for funcion in ("sin", "cos")],
                        dtype=object)


# Verificación de ImputadorLluvias con un ejemplo construido a mano
# Observados: mes 1 → 100; mes 2 → 300 y 2000; mes 12 → 150. El mes 3 no tiene valores observados.
ejemplo_lluvias = pd.DataFrame({"lluvias": [100, np.nan, 300, np.nan, 2000, 150],
                                "mes": [1, 1, 2, 3, 2, 12]})
imputador_ejemplo = ImputadorLluvias().fit(ejemplo_lluvias)
# Mediana general de los observados: (150 + 300) / 2 = 225. Percentil 99 (interpolación lineal): 300 + 0.97 × 1700 = 1949
assert imputador_ejemplo.mediana_mes_ == {1: 100, 2: 1150, 12: 150}, imputador_ejemplo.mediana_mes_
assert imputador_ejemplo.mediana_global_ == 225 and np.isclose(imputador_ejemplo.tope_, 1949)
lluvias_esperadas = [100, 100, 300, 225, 1949, 150]  # fila 2: mediana del mes 1; fila 4: mes sin datos → 225; fila 5: tope
assert np.allclose(imputador_ejemplo.transform(ejemplo_lluvias).ravel(), lluvias_esperadas)
print("ImputadorLluvias:", imputador_ejemplo.transform(ejemplo_lluvias).ravel().round(1).tolist())

# Verificación de ArmonicosMes: es una transformación sin estado, así que se compara con los armónicos del EDA
armonicos = ArmonicosMes().fit_transform(df_ipsa[[PREDICTOR_MES]])
columnas_armonicos = list(ArmonicosMes().get_feature_names_out())
assert columnas_armonicos == ["mes_sin_12", "mes_cos_12", "mes_sin_6", "mes_cos_6"]
assert np.allclose(armonicos, df_ipsa[columnas_armonicos].to_numpy()), "Los armónicos no coinciden con los del EDA"
print("ArmonicosMes: coincide con los armónicos del EDA en las", len(df_ipsa), "cosechas")

ImputadorLluvias: [100.0, 100.0, 300.0, 225.0, 1949.0, 150.0]
ArmonicosMes: coincide con los armónicos del EDA en las 2187 cosechas


### 2.2 Preprocesador completo

El preprocesador combina los dos transformadores de 2.1 con la imputación por mediana, las indicadoras de la tenencia y el escalado opcional. Se construye con una función porque se usa con y sin escalado, y con variables adicionales en las pruebas de robustez.

Se verifica con un ejemplo de seis cosechas construido a mano que la matriz de diseño tenga las 11 columnas esperadas, que no queden faltantes, que las medianas de `semsmad` y `dosismad` sean las del ejemplo, que el código 1 de la tenencia sea la referencia y que, con escalado, cada columna tenga media 0 y desviación 1.

In [22]:
# 2.2 Preprocesador completo: lluvias, numéricas, armónicos del mes, indicadoras de tenencia y escalado opcional
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CODIGOS_TENENCIA = [1, 2, 3]  # fijos de antemano: todos los pliegues producen las mismas indicadoras
NUMERICAS_MEDIANA = [columna for columna in PREDICTORES_NUMERICOS if columna != "lluvias"]


def crear_preprocesador(numericas_extra=(), escalar=True):
    """Imputación, codificación y escalado opcional. Todos los estadísticos se ajustan con los datos de fit().

    numericas_extra: variables numéricas adicionales, imputadas con la mediana (pruebas de robustez).
    """
    columnas = ColumnTransformer(
        [
            ("lluvias", ImputadorLluvias(), ["lluvias", PREDICTOR_MES]),
            ("numericas", SimpleImputer(strategy="median"), NUMERICAS_MEDIANA + list(numericas_extra)),
            ("mes", ArmonicosMes(), [PREDICTOR_MES]),
            ("tenencia", OneHotEncoder(categories=[CODIGOS_TENENCIA], drop="first", sparse_output=False),
             PREDICTORES_CATEGORICOS),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )
    pasos = [("columnas", columnas)]
    if escalar:
        pasos.append(("escalado", StandardScaler()))
    return Pipeline(pasos).set_output(transform="pandas")


# Verificación con un ejemplo de seis cosechas construido a mano
ejemplo = pd.DataFrame({
    "edad": [12.0, 13.5, 11.0, 14.0, 12.5, 13.0],
    "cortes": [1, 2, 3, 4, 2, 5],
    "semsmad": [8.0, np.nan, 10.0, 6.0, 9.0, 12.0],     # mediana de los observados: 9.0
    "lluvias": [100, np.nan, 300, np.nan, 2000, 150],   # el mismo ejemplo de 2.1
    "dosismad": [0.8, 1.0, np.nan, 1.2, 0.8, 1.0],      # mediana de los observados: 1.0
    "mes": [1, 1, 2, 3, 2, 12],
    "grupo_tenencia": [1, 2, 3, 3, 2, 1],
})
COLUMNAS_DISENO = ["lluvias", *NUMERICAS_MEDIANA, *columnas_armonicos, "grupo_tenencia_2", "grupo_tenencia_3"]

# Sin escalado: se pueden leer los valores imputados y las indicadoras
sin_escalar = crear_preprocesador(escalar=False).fit_transform(ejemplo)
assert list(sin_escalar.columns) == COLUMNAS_DISENO, list(sin_escalar.columns)
assert sin_escalar.notna().all().all(), "Quedaron faltantes"
assert sin_escalar.loc[1, "semsmad"] == 9.0 and sin_escalar.loc[2, "dosismad"] == 1.0, "Medianas incorrectas"
assert np.allclose(sin_escalar["lluvias"], lluvias_esperadas)
# El código 1 es la referencia: sus filas tienen 0 en las dos indicadoras
codigo_1 = ejemplo["grupo_tenencia"] == 1
assert (sin_escalar.loc[codigo_1, ["grupo_tenencia_2", "grupo_tenencia_3"]] == 0).all().all()

# Con escalado: media 0 y desviación 1 en cada columna del ajuste
escalado = crear_preprocesador().fit_transform(ejemplo)
assert np.allclose(escalado.mean(), 0) and np.allclose(escalado.std(ddof=0), 1), "El escalado no es el esperado"

# Variables adicionales para las pruebas de robustez: se agregan como columnas numéricas
con_extra = crear_preprocesador(numericas_extra=["pct_diatrea"], escalar=False).fit_transform(
    ejemplo.assign(pct_diatrea=[3.0, np.nan, 4.5, 2.0, 6.0, 5.0]))
assert "pct_diatrea" in con_extra.columns and con_extra.shape[1] == len(COLUMNAS_DISENO) + 1

print(f"Matriz de diseño: {len(COLUMNAS_DISENO)} columnas → {', '.join(COLUMNAS_DISENO)}")
display(sin_escalar.round(3))

Matriz de diseño: 11 columnas → lluvias, edad, cortes, semsmad, dosismad, mes_sin_12, mes_cos_12, mes_sin_6, mes_cos_6, grupo_tenencia_2, grupo_tenencia_3


,lluvias,edad,cortes,semsmad,dosismad,mes_sin_12,mes_cos_12,mes_sin_6,mes_cos_6,grupo_tenencia_2,grupo_tenencia_3
0,100.0,12.0,1.0,8.0,0.8,0.500,0.866,0.866,0.5,0.0,0.0
1,100.0,13.5,2.0,9.0,1.0,0.500,0.866,0.866,0.5,1.0,0.0
2,300.0,11.0,3.0,10.0,1.0,0.866,0.500,0.866,-0.5,0.0,1.0
3,225.0,14.0,4.0,6.0,1.2,1.000,0.000,0.000,-1.0,0.0,1.0
4,1949.0,12.5,2.0,9.0,0.8,0.866,0.500,0.866,-0.5,1.0,0.0
5,150.0,13.0,5.0,12.0,1.0,-0.000,1.000,-0.000,1.0,0.0,0.0


### Conclusión de la sección 2

Los transformadores reproducen los valores calculados a mano en el ejemplo:

- `ImputadorLluvias` reemplaza el faltante del mes 1 por la mediana de ese mes (100 mm). Usa la mediana general de los valores observados (225 mm) en el mes 3, que no tiene datos en el ajuste, y acota el valor extremo de 2000 mm en el percentil 99 de los observados (1949 mm). Los demás valores observados no cambian.
- `ArmonicosMes` coincide con los armónicos calculados en el EDA en las 2187 cosechas. Como es una transformación sin estado, calcularlos dentro del `Pipeline` no cambia los datos que recibe el modelo y permite tratar el mes como una sola variable al interpretar.

El preprocesador completo produce una matriz de diseño de 11 columnas, sin faltantes:

| Grupo | Columnas |
|---|---|
| Lluvia imputada y acotada | `lluvias` |
| Numéricas imputadas con la mediana | `edad`, `cortes`, `semsmad`, `dosismad` |
| Armónicos del mes | `mes_sin_12`, `mes_cos_12`, `mes_sin_6`, `mes_cos_6` |
| Indicadoras de tenencia (referencia: código 1) | `grupo_tenencia_2`, `grupo_tenencia_3` |

En el ejemplo, `semsmad` y `dosismad` se imputan con sus medianas (9.0 semanas y 1.0 L/ha), las cosechas con código de tenencia 1 tienen 0 en las dos indicadoras y, con escalado, cada columna queda con media 0 y desviación 1. El preprocesador también acepta variables adicionales para las pruebas de robustez.

Todos los estadísticos (medianas, tope de la lluvia, media y desviación del escalado) se calculan en `fit()`. Dentro de la validación cruzada se recalculan en cada pliegue con sus datos de ajuste, y el conjunto de prueba recibe los valores aprendidos en el entrenamiento. Las verificaciones con los datos reales se hacen en la sección 3, una vez definido el conjunto de entrenamiento.

## 3. Partición, pliegues de validación cruzada y auditoría de fuga de información

La partición tiene que reproducir la forma en que se usaría el modelo. En el escenario de uso definido al inicio, el Ingenio clasifica las cosechas que programa para los próximos periodos, antes de que ocurran. En ese momento no se conoce el resultado de ninguna cosecha de esos periodos, pero sí el de las cosechas anteriores, incluidas las de las mismas suertes en ciclos previos: 629 de las 1115 suertes del dataset tienen dos o más cosechas.

El análisis de regresión del histórico de suertes partió los datos por hacienda, porque allí el modelo tenía que generalizar a fincas nuevas, como las de proveedores que se incorporan. Su prueba de robustez (sección 6.2 de ese notebook) mostró además otra cosa: cuando el ajuste incluye cosechas de los mismos periodos que se validan, el desempeño de los modelos flexibles se infla, porque aprenden condiciones propias de cada periodo que no existen para un periodo futuro. En este dataset el riesgo de compartir periodos se revisa primero con los datos (3.1), y con ese resultado se define la agrupación de la partición.

La sección tiene cinco partes:

1. **Cosechas simultáneas (3.1):** qué tanto se parecen las cosechas de un mismo periodo.
2. **Partición en entrenamiento y prueba (3.2).**
3. **Verificación del preprocesamiento con el entrenamiento (3.3):** faltantes, escalado, VIF y reproducción de la validación cruzada de AA I.
4. **Pliegues de validación cruzada (3.4).**
5. **Auditoría de fuga de información (3.5).**

### 3.1 Cosechas simultáneas

Una hacienda puede cosechar varias de sus suertes en el mismo periodo (`periodo`, año y mes). Esas cosechas comparten el mes de cosecha, y pueden tener valores parecidos de los demás predictores y resultados parecidos, por compartir suelo, manejo y clima. Si una de ellas queda en entrenamiento y otra en prueba, el modelo se evalúa con la ayuda de una cosecha simultánea cuyo resultado no se conocería al clasificar ese periodo.

Para medir qué tan frecuente es esa situación, se calcula la proporción de cosechas que tienen al menos otra suerte de su hacienda cosechada en el mismo periodo. Después se comparan todas las parejas de cosechas de suertes distintas, separadas en tres tipos:

- de la misma hacienda y el mismo periodo;
- de otra hacienda en el mismo periodo;
- de periodos distintos.

Para cada tipo se calcula el porcentaje de parejas con la misma lluvia registrada, la diferencia mediana de edad y de número de cortes, y el porcentaje de parejas con la misma clase de `TCH` y de sacarosa. Ese último se compara con la coincidencia que se esperaría por azar, $\sum_c p_c^2$, donde $p_c$ es la proporción de la clase $c$.

Se usan las 2187 cosechas. La lluvia solo se compara cuando está medida en las dos cosechas de la pareja, y la clase de `TCH`, solo cuando ambas tienen clase. Las parejas de cosechas de una misma suerte se excluyen, porque pertenecen necesariamente a periodos distintos y su parecido es el que se espera de un mismo lote.

In [23]:
# 3.1 Cosechas simultáneas: cosechas de una misma hacienda en el mismo periodo
# Cosechas de cada hacienda en cada periodo, sobre las 2187 cosechas
tamano_grupo = df_ipsa.groupby(["FAZ", "periodo"])["suerte"].transform("size")
con_simultanea = tamano_grupo > 1
print(f"Cosechas con al menos otra suerte de su hacienda en el mismo periodo: {con_simultanea.sum()} de "
      f"{len(df_ipsa)} ({100 * con_simultanea.mean():.1f} %)")
print(f"Suertes cosechadas por hacienda y periodo: mediana "
      f"{df_ipsa.groupby(['FAZ', 'periodo']).size().median():.0f} | máximo {df_ipsa.groupby(['FAZ', 'periodo']).size().max()}")

# Todas las parejas de cosechas (i < j), solo con las columnas necesarias para no ocupar memoria de más
COLUMNAS_PAREJAS = ["FAZ", "periodo", "suerte", "lluvias", "edad", "cortes", *COLUMNA_CLASE.values()]
fila_i, fila_j = np.triu_indices(len(df_ipsa), k=1)
cosecha_a = df_ipsa[COLUMNAS_PAREJAS].iloc[fila_i].reset_index(drop=True)
cosecha_b = df_ipsa[COLUMNAS_PAREJAS].iloc[fila_j].reset_index(drop=True)

# Se excluyen las parejas de una misma suerte
distinta_suerte = (cosecha_a["suerte"] != cosecha_b["suerte"]).to_numpy()
cosecha_a = cosecha_a[distinta_suerte].reset_index(drop=True)
cosecha_b = cosecha_b[distinta_suerte].reset_index(drop=True)

TIPOS_PAREJA = ["Misma hacienda y mismo periodo", "Otra hacienda, mismo periodo", "Distinto periodo"]
mismo_periodo = cosecha_a["periodo"] == cosecha_b["periodo"]
tipo_pareja = pd.Series(np.select([mismo_periodo & (cosecha_a["FAZ"] == cosecha_b["FAZ"]), mismo_periodo],
                                  TIPOS_PAREJA[:2], default=TIPOS_PAREJA[2]), name="Tipo de pareja")


def porcentaje_igual(columna):
    """Porcentaje de parejas con el mismo valor, entre las que tienen el dato en las dos cosechas."""
    validas = cosecha_a[columna].notna() & cosecha_b[columna].notna()
    igual = (cosecha_a[columna].astype(object) == cosecha_b[columna].astype(object)).astype(float).where(validas)
    return igual.groupby(tipo_pareja).mean() * 100


parejas = pd.DataFrame({
    "Parejas": tipo_pareja.value_counts(),
    "% misma lluvia": porcentaje_igual("lluvias"),
    "Dif. mediana de edad (meses)": (cosecha_a["edad"] - cosecha_b["edad"]).abs().groupby(tipo_pareja).median(),
    "Dif. mediana de cortes": (cosecha_a["cortes"] - cosecha_b["cortes"]).abs().groupby(tipo_pareja).median(),
    "% misma clase TCH": porcentaje_igual("clase_tch_ingenio"),
    "% misma clase sacarosa": porcentaje_igual("clase_sac_ingenio"),
}).reindex(TIPOS_PAREJA)
display(parejas.round(1))

# Coincidencia de clase esperada por azar: probabilidad de que dos cosechas tomadas al azar tengan la misma clase
coincidencia_azar = {target: (df_ipsa[columna].value_counts(normalize=True) ** 2).sum() * 100
                     for target, columna in COLUMNA_CLASE.items()}
print(f"Coincidencia de clase esperada por azar: TCH {coincidencia_azar['TCH']:.1f} % | "
      f"sacarosa {coincidencia_azar['sacarosa']:.1f} %")

# Se liberan las tablas de parejas (más de dos millones de filas cada una)
del cosecha_a, cosecha_b, fila_i, fila_j, tipo_pareja, mismo_periodo, distinta_suerte

Cosechas con al menos otra suerte de su hacienda en el mismo periodo: 1365 de 2187 (62.4 %)
Suertes cosechadas por hacienda y periodo: mediana 1 | máximo 12


,Parejas,% misma lluvia,Dif. mediana de edad (meses),Dif. mediana de cortes,% misma clase TCH,% misma clase sacarosa
Tipo de pareja,,,,,,
Misma hacienda y mismo periodo,1655,34.5,0.2,0.0,58.3,56.0
"Otra hacienda, mismo periodo",53174,0.7,0.7,1.0,36.3,48.1
Distinto periodo,2333868,0.3,0.9,1.0,35.0,35.2


Coincidencia de clase esperada por azar: TCH 35.1 % | sacarosa 35.6 %


#### Conclusión de 3.1

El 62.4 % de las cosechas (1365 de 2187) tiene al menos otra suerte de su hacienda cosechada en el mismo periodo. La mediana es de una suerte por hacienda y periodo, pero algunas haciendas cosechan hasta 12 suertes en un mismo mes.

| Tipo de pareja | Parejas | Misma lluvia | Dif. mediana de edad | Dif. mediana de cortes | Misma clase `TCH` | Misma clase sacarosa |
|---|---|---|---|---|---|---|
| Misma hacienda y mismo periodo | 1655 | 34.5 % | 0.2 meses | 0 | 58.3 % | 56.0 % |
| Otra hacienda, mismo periodo | 53 174 | 0.7 % | 0.7 meses | 1 | 36.3 % | 48.1 % |
| Distinto periodo | 2 333 868 | 0.3 % | 0.9 meses | 1 | 35.0 % | 35.2 % |

La coincidencia de clase esperada por azar es 35.1 % en `TCH` y 35.6 % en sacarosa.

- **Las cosechas simultáneas de una misma hacienda son casi duplicadas en sus predictores.** Su edad difiere en una mediana de 0.2 meses, tienen el mismo número de corte y el 34.5 % registra exactamente la misma lluvia, frente a menos del 1 % en los demás tipos de pareja. Que la lluvia no coincida en la mayoría de ellas indica que `lluvias` no es un valor común de la hacienda en el mes, sino que se acumula en una ventana propia de cada suerte.
- **En `TCH`, el parecido es propio de cada hacienda.** La clase coincide en el 58.3 % de las parejas de la misma hacienda y el mismo periodo, frente al 35.0–36.3 % de los otros dos tipos, que es prácticamente el nivel del azar.
- **En sacarosa, el parecido es propio del periodo.** La coincidencia sube tanto en la misma hacienda (56.0 %) como en otras haciendas del mismo periodo (48.1 %), frente al 35.2 % entre periodos distintos. La sacarosa depende de condiciones que comparten todas las haciendas en un mismo mes, coherente con su estacionalidad.

Con una partición aleatoria, o agrupada solo por suerte, la mayoría de las cosechas de prueba tendría en entrenamiento una cosecha simultánea casi idéntica en sus predictores y con una clase parecida, cuyo resultado no se conocería al clasificar ese periodo. Agrupar por hacienda evitaría las parejas de la misma hacienda, pero no las de otras haciendas del mismo periodo, que se parecen en sacarosa. Por eso la partición se agrupa por periodo: todas las cosechas de un mismo mes y año quedan en el mismo conjunto. Es la misma lección de la regresión del histórico de suertes, donde compartir periodos entre ajuste y validación inflaba el desempeño.

### 3.2 Partición en entrenamiento y prueba

La partición cumple tres condiciones:

- **Agrupada por periodo.** Todas las cosechas de un mismo mes y año quedan en el mismo conjunto. Así, el desempeño en prueba corresponde a periodos que el modelo no vio, sin cosechas simultáneas en entrenamiento.
- **Estratificada.** Conserva en los dos conjuntos, en lo posible, la proporción de cada combinación de clases de `TCH` y de sacarosa. Los 4 registros con `excluir_tch` tienen `TCH` menor que 60 t/ha, así que, solo para estratificar, se agrupan con la clase baja de `TCH`.
- **Común a los dos targets.** Los dos problemas se evalúan sobre los mismos periodos de prueba.

Se usa `StratifiedGroupKFold` con 5 pliegues, y uno de ellos se toma como conjunto de prueba, lo que da una proporción cercana al 20 %. Como cada periodo agrupa muchas cosechas, el conjunto de prueba queda formado por pocos periodos completos y puede dejar sin representar algunos meses del año. El mes de cosecha es uno de los predictores con fundamento agronómico más claro, así que el pliegue se elige con una regla fija que solo mira la composición del conjunto de prueba: el de menor número de meses con menos de 10 cosechas y, en caso de empate, el de menor diferencia en la proporción de clases entre entrenamiento y prueba. La regla no usa el desempeño de ningún modelo.

Es la misma partición de AA I (`02_modelos_clasificacion.ipynb`), con la misma semilla y la misma regla. Se verifica con `assert` que el notebook la reproduce: el pliegue elegido, el número de registros y el número de periodos de cada conjunto. En AA I, este conjunto de prueba se usó para evaluar la regresión logística y KNN. Ninguna decisión de este notebook se toma con él: los predictores, los hiperparámetros y la regla de elección se definen solo con el entrenamiento, y el conjunto de prueba se usa una sola vez, en la sección 8.

Además del tamaño y del balance de clases, se revisa cómo se distribuyen los años y los meses de cosecha en cada conjunto.

In [24]:
# 3.2 Partición en entrenamiento y prueba: agrupada por periodo, estratificada y común a los dos targets
from sklearn.model_selection import StratifiedGroupKFold

# Partición de AA I (02_modelos_clasificacion.ipynb, sección 11.2): se verifica que se reproduce
PARTICION_AA1 = {"pliegue_prueba": 3, "registros_train": 1737, "registros_test": 450,
                 "periodos_train": 56, "periodos_test": 16}

# Estrato: combinación de las clases de ambos targets. Los 4 registros con excluir_tch (TCH < 60 t/ha)
# se agrupan con la clase baja de TCH solo para estratificar; no reciben clase de TCH
estrato = (df_ipsa["clase_tch_ingenio"].astype(str).where(~df_ipsa["excluir_tch"], "bajo")
           + "|" + df_ipsa["clase_sac_ingenio"].astype(str))

particionador = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
pliegues_candidatos = list(particionador.split(df_ipsa, estrato, groups=df_ipsa["periodo"]))


def diferencia_clases(entrenamiento, prueba):
    """Máxima diferencia, en puntos porcentuales, entre la proporción de una clase en entrenamiento y en prueba,
    considerando los dos targets (los registros sin clase no cuentan)."""
    return max((entrenamiento[columna].value_counts(normalize=True)
                - prueba[columna].value_counts(normalize=True)).abs().max()
               for columna in COLUMNA_CLASE.values()) * 100


# Características de cada pliegue candidato a conjunto de prueba
candidatos = pd.DataFrame([{
    "% prueba": len(prueba) / len(df_ipsa) * 100,
    "Periodos en prueba": df_ipsa.iloc[prueba]["periodo"].nunique(),
    "Meses con menos de 10 cosechas en prueba": (df_ipsa.iloc[prueba][PREDICTOR_MES].value_counts()
                                                 .reindex(range(1, 13), fill_value=0) < 10).sum(),
    "Años en prueba": df_ipsa.iloc[prueba]["anio"].nunique(),
    "Máx. diferencia de clase (pp)": diferencia_clases(df_ipsa.iloc[entrenamiento], df_ipsa.iloc[prueba]),
} for entrenamiento, prueba in pliegues_candidatos]).rename_axis("Pliegue")
display(candidatos.round(1))

# Regla fija: menos meses mal representados en prueba; en empate, menor diferencia de clases
pliegue_prueba = candidatos.sort_values(["Meses con menos de 10 cosechas en prueba",
                                         "Máx. diferencia de clase (pp)"]).index[0]
indices_train, indices_test = pliegues_candidatos[pliegue_prueba]
df_train, df_test = df_ipsa.iloc[indices_train].copy(), df_ipsa.iloc[indices_test].copy()
print(f"Pliegue elegido como conjunto de prueba: {pliegue_prueba}")

# Ningún periodo puede estar en los dos conjuntos
assert set(df_train["periodo"]).isdisjoint(df_test["periodo"]), "Hay periodos en entrenamiento y en prueba"

# Reproducción de la partición de AA I
assert pliegue_prueba == PARTICION_AA1["pliegue_prueba"], "El pliegue elegido no es el de AA I"
assert (len(df_train), len(df_test)) == (PARTICION_AA1["registros_train"], PARTICION_AA1["registros_test"])
assert (df_train["periodo"].nunique(), df_test["periodo"].nunique()) == \
       (PARTICION_AA1["periodos_train"], PARTICION_AA1["periodos_test"])

# Tamaño de cada conjunto, en total y por target (los registros sin clase de TCH no cuentan para TCH)
for nombre, datos in [("Entrenamiento", df_train), ("Prueba", df_test)]:
    print(f"{nombre}: {len(datos)} registros ({100 * len(datos) / len(df_ipsa):.1f} % del total)")
resumen_particion = pd.DataFrame({
    nombre: {"Registros": len(datos), "Periodos": datos["periodo"].nunique(), "Suertes": datos["suerte"].nunique(),
             "Haciendas": datos["FAZ"].nunique(), "Con excluir_tch": int(datos["excluir_tch"].sum()),
             "Registros de TCH": int(datos["clase_tch_ingenio"].notna().sum()),
             "Registros de sacarosa": int(datos["clase_sac_ingenio"].notna().sum())}
    for nombre, datos in [("Entrenamiento", df_train), ("Prueba", df_test)]
})
display(resumen_particion)

# Proporción de cada clase en entrenamiento y en prueba, por target
balance_particion = pd.DataFrame({
    (target, nombre): datos[columna].value_counts(normalize=True).reindex(CLASES) * 100
    for target, columna in COLUMNA_CLASE.items()
    for nombre, datos in [("entrenamiento", df_train), ("prueba", df_test)]
}).round(1)
display(balance_particion)

# Distribución de las cosechas por año y por mes de cosecha en cada conjunto
for variable, etiqueta in [("anio", "Año"), (PREDICTOR_MES, "Mes")]:
    display(pd.DataFrame({"Entrenamiento": df_train[variable].value_counts(),
                          "Prueba": df_test[variable].value_counts()})
            .sort_index().fillna(0).astype(int).rename_axis(etiqueta).T)

,% prueba,Periodos en prueba,Meses con menos de 10 cosechas en prueba,Años en prueba,Máx. diferencia de clase (pp)
Pliegue,,,,,
0,19.6,12,5,5,3.0
1,19.8,15,6,7,0.7
2,19.9,15,4,7,2.3
3,20.6,16,4,6,0.3
4,20.2,14,6,5,0.8


Pliegue elegido como conjunto de prueba: 3
Entrenamiento: 1737 registros (79.4 % del total)
Prueba: 450 registros (20.6 % del total)


,Entrenamiento,Prueba
Registros,1737,450
Periodos,56,16
Suertes,1000,401
Haciendas,262,171
Con excluir_tch,4,0
Registros de TCH,1733,450
Registros de sacarosa,1737,450


TCH             sacarosa       
      entrenamiento prueba entrenamiento prueba
bajo           22.6   22.7          21.5   21.6
medio          38.9   39.1          36.6   36.9
alto           38.5   38.2          41.9   41.6

Año,2014,2015,2016,2017,2018,2019,2020,2021
Entrenamiento,1,28,89,190,244,522,577,86
Prueba,0,7,21,27,164,80,151,0


Mes,1,2,3,4,5,6,7,8,9,10,11,12
Entrenamiento,216,67,98,82,173,175,90,104,167,249,126,190
Prueba,15,3,87,98,1,44,36,45,54,5,62,0


#### Conclusión de 3.2

| Pliegue | % prueba | Periodos en prueba | Meses con menos de 10 cosechas en prueba | Años en prueba | Máx. diferencia de clase (pp) |
|---|---|---|---|---|---|
| 0 | 19.6 | 12 | 5 | 5 | 3.0 |
| 1 | 19.8 | 15 | 6 | 7 | 0.7 |
| 2 | 19.9 | 15 | 4 | 7 | 2.3 |
| **3** | **20.6** | **16** | **4** | **6** | **0.3** |
| 4 | 20.2 | 14 | 6 | 5 | 0.8 |

- Se eligió el pliegue 3. Comparte con el pliegue 2 el menor número de meses mal representados (4), y tiene la menor diferencia en la proporción de clases entre entrenamiento y prueba (0.3 puntos porcentuales). La partición reproduce la de AA I: el mismo pliegue, el mismo número de registros y el mismo número de periodos en cada conjunto.

| | Entrenamiento | Prueba |
|---|---|---|
| Registros | 1737 (79.4 %) | 450 (20.6 %) |
| Periodos | 56 | 16 |
| Suertes | 1000 | 401 |
| Haciendas | 262 | 171 |
| Registros con `excluir_tch` | 4 | 0 |
| Registros de `TCH` | 1733 | 450 |
| Registros de sacarosa | 1737 | 450 |

- La proporción es 79.4/20.6 y no exactamente 80/20, porque cada conjunto está formado por periodos completos. Ningún periodo aparece en los dos conjuntos, así que el desempeño en prueba corresponde a meses que el modelo no vio, sin cosechas simultáneas en entrenamiento.
- Los 4 registros con `excluir_tch` quedaron en entrenamiento. La clasificación de `TCH` usa 1733 registros de entrenamiento y los 450 de prueba; la de sacarosa, 1737 y 450.
- La proporción de cada clase difiere como máximo 0.3 puntos porcentuales entre los dos conjuntos:

| Clase | `TCH` entrenamiento | `TCH` prueba | Sacarosa entrenamiento | Sacarosa prueba |
|---|---|---|---|---|
| Bajo | 22.6 % | 22.7 % | 21.5 % | 21.6 % |
| Medio | 38.9 % | 39.1 % | 36.6 % | 36.9 % |
| Alto | 38.5 % | 38.2 % | 41.9 % | 41.6 % |

- El conjunto de prueba cubre cosechas de 2015 a 2020. No incluye 2014 ni 2021, que tienen pocas cosechas en total (1 y 86).
- Ningún pliegue candidato cubre bien los 12 meses, porque el conjunto de prueba está formado por pocos periodos. En el elegido, febrero (3 cosechas), mayo (1), octubre (5) y diciembre (0) quedan con menos de 10 cosechas en prueba. El entrenamiento tiene al menos 67 cosechas en cada mes, así que los modelos aprenden el efecto de todos los meses, pero la evaluación final no mide su desempeño en esos cuatro.
- Entre los dos conjuntos suman 433 haciendas, más que las 285 del dataset, así que muchas haciendas tienen cosechas en ambos, en periodos distintos. Es lo esperado al agrupar por periodo, y corresponde al escenario de uso: el Ingenio clasifica nuevos periodos de haciendas y suertes que ya conoce. En la auditoría de 3.5 se cuenta cuántas cosechas de prueba pertenecen a suertes con cosechas en entrenamiento.

### 3.3 Verificación del preprocesamiento con el entrenamiento

Con el conjunto de entrenamiento definido, el preprocesador de la sección 2 se verifica con los datos reales en tres pasos.

**Ajuste y transformación.** Se ajusta con el entrenamiento y se transforman los dos conjuntos. Se verifica que no queden faltantes, que el escalado deje cada columna de entrenamiento con media 0 y desviación 1, y que los 12 meses tengan lluvia observada en el entrenamiento, de modo que ninguna imputación tenga que usar la mediana general. Se reporta cuántos registros se imputan y cuántos se acotan en cada conjunto.

**VIF de la matriz de diseño.** El factor de inflación de la varianza de la columna $j$ es

$$\mathrm{VIF}_j = \frac{1}{1 - R_j^2},$$

donde $R_j^2$ es el coeficiente de determinación de la regresión de esa columna sobre todas las demás. Un VIF mayor que 5 indica una multicolinealidad moderada, y uno mayor que 10, una severa. Aunque se asocia a la regresión lineal, también importa aquí: la regresión logística es lineal en el logit, así que la multicolinealidad hace inestables sus coeficientes y dificulta interpretarlos. Se calcula sobre la matriz de diseño de entrenamiento, que ahora incluye la lluvia imputada, los armónicos y las indicadoras. Escalar las columnas no cambia el VIF.

**Reproducción de la validación cruzada de AA I.** El preprocesador reescribe el de AA I con un cambio: los armónicos del mes se calculan dentro del `Pipeline`. Para confirmar que aplica las mismas reglas, se reproduce el F1 macro de validación cruzada de los modelos finales de AA I con los umbrales del Ingenio (la regresión logística y KNN), con sus mismos predictores, hiperparámetros y pliegues:

- **Predictores:** los 7 de este notebook más `pct_diatrea`, en el orden de columnas de AA I.
- **Hiperparámetros:** en `TCH`, logística L1 con ponderación balanceada y $C = 10^{0.5} \approx 3.16$, y KNN uniforme con $k = 13$. En sacarosa, logística L1 sin ponderar con $C = 10$, y KNN ponderado por distancia con $k = 9$.
- **Pliegues:** un `StratifiedGroupKFold` de 5 pliegues sobre el entrenamiento, agrupado por periodo y estratificado por la clase de cada target con sus etiquetas de texto, como en AA I.

El F1 macro (la media armónica de precision y recall, promediada sobre las tres clases) se define en la sección 4. Los pliegues de AA I se usan solo en esta verificación; los de este notebook se definen en 3.4. AA I reportó los valores con tres decimales, así que se acepta una diferencia de hasta 0.0005.

In [25]:
# 3.3 Verificación del preprocesamiento con el entrenamiento: faltantes, escalado, imputación de lluvias y VIF
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

preprocesador_train = crear_preprocesador().fit(df_train[PREDICTORES])
X_train_diseno = preprocesador_train.transform(df_train[PREDICTORES])
X_test_diseno = preprocesador_train.transform(df_test[PREDICTORES])  # solo se verifica su estructura

assert list(X_train_diseno.columns) == COLUMNAS_DISENO, "Columnas inesperadas en la matriz de diseño"
assert X_train_diseno.notna().all().all() and X_test_diseno.notna().all().all(), "Quedaron faltantes"
assert np.allclose(X_train_diseno.mean(), 0) and np.allclose(X_train_diseno.std(ddof=0), 1), \
    "El escalado no se ajustó con el entrenamiento"

# Imputación de lluvias: todos los meses deben tener lluvia observada en el entrenamiento
imputador_train = preprocesador_train.named_steps["columnas"].named_transformers_["lluvias"]
assert set(imputador_train.mediana_mes_) == set(range(1, 13)), "Hay meses sin lluvia observada en el entrenamiento"
print(f"Lluvias: mediana general {imputador_train.mediana_global_:.1f} mm | "
      f"tope (P{imputador_train.percentil_tope}) {imputador_train.tope_:.1f} mm")
display(pd.DataFrame({
    "Imputados": [df_train["lluvias"].isna().sum(), df_test["lluvias"].isna().sum()],
    "% imputados": [100 * df_train["lluvias"].isna().mean(), 100 * df_test["lluvias"].isna().mean()],
    "Acotados al tope": [(df_train["lluvias"] > imputador_train.tope_).sum(),
                         (df_test["lluvias"] > imputador_train.tope_).sum()],
}, index=["Entrenamiento", "Prueba"]).round(1))

# Mediana usada para imputar en cada mes de cosecha (una fila por mes, para conservar los conteos como enteros)
display(pd.DataFrame({
    "Cosechas (entrenamiento)": df_train.groupby(PREDICTOR_MES).size(),
    "Con lluvia medida": df_train.groupby(PREDICTOR_MES)["lluvias"].count(),
    "Mediana imputada (mm)": pd.Series(imputador_train.mediana_mes_),
}).rename_axis("Mes"))

# VIF de cada columna de la matriz de diseño de entrenamiento (statsmodels requiere la constante)
X_vif = sm.add_constant(X_train_diseno)
vif_diseno = pd.Series([variance_inflation_factor(X_vif.to_numpy(), posicion)
                        for posicion in range(1, X_vif.shape[1])],
                       index=X_train_diseno.columns, name="VIF").sort_values(ascending=False)
display(vif_diseno.round(2).to_frame())

Lluvias: mediana general 132.5 mm | tope (P99) 748.1 mm


,Imputados,% imputados,Acotados al tope
Entrenamiento,571,32.9,12
Prueba,114,25.3,0


,Cosechas (entrenamiento),Con lluvia medida,Mediana imputada (mm)
Mes,,,
1,216,129,134.0
2,67,56,75.0
3,98,61,101.0
4,82,45,198.0
5,173,99,181.0
6,175,139,172.0
7,90,71,109.0
8,104,68,129.5
9,167,103,74.0


,VIF
grupo_tenencia_3,2.82
edad,2.55
grupo_tenencia_2,2.49
semsmad,2.21
lluvias,1.90
dosismad,1.30
mes_sin_12,1.20
cortes,1.20
mes_cos_6,1.20
mes_cos_12,1.10


In [26]:
# 3.3 Reproducción del F1 macro de validación cruzada de los modelos finales de AA I (umbrales del Ingenio)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.neighbors import KNeighborsClassifier

# Resultados de AA I (02_modelos_clasificacion.ipynb, sección 17.1, columna "F1 CV"), con tres decimales
F1_CV_AA1 = {("TCH", "Logística"): 0.422, ("TCH", "KNN"): 0.378,
             ("sacarosa", "Logística"): 0.448, ("sacarosa", "KNN"): 0.443}
TOLERANCIA_F1 = 5e-4 + 1e-9  # los valores de AA I están redondeados a tres decimales

# Hiperparámetros de los modelos finales de AA I. C sale de su grilla logspace(-3, 2, 11): índices 7 (10^0.5) y 8 (10)
GRILLA_C_AA1 = np.logspace(-3, 2, 11)
MODELOS_AA1 = {
    ("TCH", "Logística"): LogisticRegression(solver="saga", max_iter=5000, random_state=RANDOM_STATE,
                                             l1_ratio=1.0, class_weight="balanced", C=GRILLA_C_AA1[7]),
    ("TCH", "KNN"): KNeighborsClassifier(n_neighbors=13, weights="uniform"),
    ("sacarosa", "Logística"): LogisticRegression(solver="saga", max_iter=5000, random_state=RANDOM_STATE,
                                                  l1_ratio=1.0, class_weight=None, C=GRILLA_C_AA1[8]),
    ("sacarosa", "KNN"): KNeighborsClassifier(n_neighbors=9, weights="distance"),
}

# Preprocesador con los predictores de AA I y su orden de columnas: lluvias, numéricas (con pct_diatrea), armónicos, tenencia
NUMERICAS_AA1 = ["edad", "cortes", "semsmad", "pct_diatrea", "dosismad"]


def crear_preprocesador_aa1():
    """Mismas reglas que crear_preprocesador, con los predictores y el orden de columnas de AA I."""
    columnas = ColumnTransformer(
        [
            ("lluvias", ImputadorLluvias(), ["lluvias", PREDICTOR_MES]),
            ("numericas", SimpleImputer(strategy="median"), NUMERICAS_AA1),
            ("mes", ArmonicosMes(), [PREDICTOR_MES]),
            ("tenencia", OneHotEncoder(categories=[CODIGOS_TENENCIA], drop="first", sparse_output=False),
             PREDICTORES_CATEGORICOS),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )
    return Pipeline([("columnas", columnas), ("escalado", StandardScaler())]).set_output(transform="pandas")


# Pliegues de AA I: un solo StratifiedGroupKFold, estratificado por la clase de cada target (etiquetas de texto)
pliegues_aa1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

filas_aa1 = []
for (target, nombre_modelo), modelo in MODELOS_AA1.items():
    columna_clase = COLUMNA_CLASE[target]
    datos = df_train[df_train[columna_clase].notna()]
    pipeline_aa1 = Pipeline([("prep", crear_preprocesador_aa1()), ("modelo", modelo)])
    resultados = cross_validate(pipeline_aa1, datos[PREDICTORES + ["pct_diatrea"]], datos[columna_clase].astype(str),
                                groups=datos["periodo"], cv=pliegues_aa1, scoring="f1_macro", n_jobs=-1)
    f1_cv = resultados["test_score"].mean()
    filas_aa1.append({"Target": target, "Modelo": nombre_modelo, "Registros": len(datos),
                      "F1 macro CV": f1_cv, "F1 macro CV (AA I)": F1_CV_AA1[(target, nombre_modelo)]})
    assert abs(f1_cv - F1_CV_AA1[(target, nombre_modelo)]) < TOLERANCIA_F1, \
        f"{target} | {nombre_modelo}: F1 = {f1_cv:.4f}, el preprocesamiento no reproduce el de AA I"

display(pd.DataFrame(filas_aa1).set_index(["Target", "Modelo"]).round(4))

Registros  F1 macro CV  F1 macro CV (AA I)
Target   Modelo                                               
TCH      Logística       1733       0.4221               0.422
         KNN             1733       0.3775               0.378
sacarosa Logística       1737       0.4478               0.448
         KNN             1737       0.4431               0.443

#### Conclusión de 3.3

**Ajuste y transformación.** Ajustado con el entrenamiento, el preprocesador produce las 11 columnas esperadas, sin faltantes en ninguno de los dos conjuntos, y el escalado deja cada columna de entrenamiento con media 0 y desviación 1.

- Se imputa la lluvia de 571 cosechas de entrenamiento (32.9 %) y 114 de prueba (25.3 %), que suman los 685 faltantes del dataset.
- Los 12 meses tienen lluvia medida en el entrenamiento (abril es el de menos, con 45 cosechas), así que todas las imputaciones usan la mediana de su mes y ninguna la mediana general. Las medianas mensuales van de 74 mm (septiembre) a 206 mm (diciembre), frente a una mediana general de 132.5 mm. Imputar con la mediana general habría asignado el mismo valor a cosechas de meses con lluvias muy distintas.
- El tope es 748.1 mm, más de cinco veces la mediana general. Se acotan 12 cosechas de entrenamiento y ninguna de prueba, así que el acotamiento solo actúa sobre los valores más extremos.

**VIF.** Todas las columnas tienen un VIF menor que 3:

| Columna | VIF |
|---|---|
| `grupo_tenencia_3` | 2.82 |
| `edad` | 2.55 |
| `grupo_tenencia_2` | 2.49 |
| `semsmad` | 2.21 |
| `lluvias` | 1.90 |
| `dosismad` | 1.30 |
| `cortes` | 1.20 |
| Armónicos del mes | 1.06 a 1.20 |

- Los VIF más altos son los de las dos indicadoras de tenencia. La referencia, el código 1, es el grupo menos frecuente, así que casi toda cosecha que no es del código 2 es del código 3, y las dos indicadoras quedan relacionadas entre sí. Es una consecuencia de la codificación, no de un solapamiento con otros predictores.
- `edad` y `semsmad` tienen los VIF más altos entre las variables continuas, coherente con su correlación en el EDA: las cosechas retrasadas alargan a la vez la edad del cultivo y el tiempo desde la aplicación del madurante.
- No hay multicolinealidad que impida interpretar los coeficientes de la regresión logística uno por uno.

**Reproducción de AA I.** El F1 macro de validación cruzada coincide con el de AA I en los cuatro modelos, dentro del redondeo a tres decimales:

| Target | Modelo | F1 macro CV | AA I |
|---|---|---|---|
| `TCH` | Logística | 0.4221 | 0.422 |
| `TCH` | KNN | 0.3775 | 0.378 |
| Sacarosa | Logística | 0.4478 | 0.448 |
| Sacarosa | KNN | 0.4431 | 0.443 |

El preprocesador aplica las mismas reglas que el de AA I, aunque ahora calcula los armónicos del mes dentro del `Pipeline`. Estos valores solo verifican el preprocesamiento: los modelos de este notebook usan otros predictores y otros pliegues, y se comparan a partir de la sección 5.

### 3.4 Pliegues de validación cruzada

Los hiperparámetros y la comparación entre modelos se deciden con validación cruzada de 5 pliegues sobre el entrenamiento, agrupada por periodo igual que la partición, para que cada pliegue de validación esté formado por meses que el modelo no vio en ese ajuste. Como cada pliegue reúne unos 11 de los 56 periodos de entrenamiento, el desempeño de un modelo puede cambiar bastante según los meses que le toquen, y una sola asignación de periodos a pliegues haría depender la comparación de esa asignación. Por eso la validación se repite con 3 asignaciones distintas de los periodos (semillas 42, 43 y 44), para un total de 15 pliegues.

Los periodos se asignan con `GroupKFold` con barajado: en cada repetición se ordenan al azar según su semilla y se reparten en 5 grupos. No se usa `StratifiedGroupKFold`, como en la partición, porque esa clase ordena los grupos según su distribución de clases antes de repartirlos, y su barajado solo cambia el orden de los grupos con una distribución casi igual [3]. Con periodos de distribuciones distintas, semillas distintas pueden producir la misma asignación, y las repeticiones dejarían de aportar información. Sin estratificación, la proporción de cada clase puede variar entre pliegues. Por eso se verifica que todo pliegue de validación tenga las tres clases, y se revisa cuánto varía la proporción de la clase baja.

Los pliegues se construyen una sola vez y se guardan. Todos los modelos, en los dos targets, se evalúan exactamente en los mismos 15 pliegues, así que las diferencias entre modelos se pueden calcular pliegue a pliegue y leer frente a la variación entre pliegues. Los dos targets comparten la asignación de periodos. En `TCH`, los 4 registros sin clase se retiran después de asignar los pliegues, sin cambiar la asignación de los demás.

Además se verifica que en cada repetición cada periodo quede completo en un solo pliegue, que ningún periodo aparezca a la vez en ajuste y validación, y que las tres asignaciones sean distintas.

In [27]:
# 3.4 Pliegues de validación cruzada: 3 repeticiones × 5 pliegues, agrupados por periodo y comunes a todos los modelos
from dataclasses import dataclass

from sklearn.model_selection import GroupKFold

N_PLIEGUES, N_REPETICIONES = 5, 3
SEMILLAS_REPETICION = [RANDOM_STATE + repeticion for repeticion in range(N_REPETICIONES)]  # 42, 43, 44

# Asignación de cada registro de entrenamiento a un pliegue de validación, en cada repetición.
# GroupKFold con barajado reparte los periodos al azar según la semilla, así que cada repetición es distinta
pliegue_asignado = pd.DataFrame(index=df_train.index)
for repeticion, semilla in enumerate(SEMILLAS_REPETICION):
    asignador = GroupKFold(n_splits=N_PLIEGUES, shuffle=True, random_state=semilla)
    asignacion = pd.Series(-1, index=df_train.index)
    for numero_pliegue, (_, posiciones_validacion) in enumerate(
            asignador.split(df_train, groups=df_train["periodo"])):
        asignacion.iloc[posiciones_validacion] = numero_pliegue
    assert (asignacion >= 0).all(), f"Repetición {repeticion}: hay registros sin pliegue"
    pliegue_asignado[f"repeticion_{repeticion}"] = asignacion

# Cada periodo debe quedar completo en un solo pliegue de cada repetición
for columna in pliegue_asignado:
    pliegues_por_periodo = pliegue_asignado[columna].groupby(df_train["periodo"]).nunique()
    assert (pliegues_por_periodo == 1).all(), f"{columna}: un periodo quedó repartido en más de un pliegue"

# Las tres asignaciones deben ser distintas: ningún par de repeticiones reparte los periodos igual
periodos_por_pliegue = {columna: {frozenset(df_train.loc[pliegue_asignado[columna] == k, "periodo"])
                                  for k in range(N_PLIEGUES)} for columna in pliegue_asignado}
columnas_repeticion = list(pliegue_asignado.columns)
for posicion, columna_a in enumerate(columnas_repeticion):
    for columna_b in columnas_repeticion[posicion + 1:]:
        assert periodos_por_pliegue[columna_a] != periodos_por_pliegue[columna_b], \
            f"{columna_a} y {columna_b} reparten los periodos igual"


@dataclass
class DatosTarget:
    """Conjuntos de entrenamiento y prueba de un target, con sus 15 pliegues de validación cruzada."""
    X_train: pd.DataFrame
    y_train: pd.Series
    periodo_train: pd.Series
    X_test: pd.DataFrame
    y_test: pd.Series
    pliegues: list  # pares (posiciones de ajuste, posiciones de validación) dentro de X_train


# Columnas que reciben los modelos: los predictores y las variables de las pruebas de robustez
COLUMNAS_ENTRADA = PREDICTORES + PREDICTORES_ROBUSTEZ

datos_por_target = {}
for target, columna_codigo in COLUMNA_CODIGO.items():
    entrenamiento = df_train[df_train[columna_codigo].notna()]
    prueba = df_test[df_test[columna_codigo].notna()]
    # En TCH se retiran los registros sin clase después de asignar los pliegues, sin cambiar la asignación
    asignacion_target = pliegue_asignado.loc[entrenamiento.index]
    pliegues_target = [
        (np.flatnonzero(asignacion_target[columna] != k), np.flatnonzero(asignacion_target[columna] == k))
        for columna in asignacion_target for k in range(N_PLIEGUES)
    ]
    datos_por_target[target] = DatosTarget(
        X_train=entrenamiento[COLUMNAS_ENTRADA], y_train=entrenamiento[columna_codigo].astype(int),
        periodo_train=entrenamiento["periodo"],
        X_test=prueba[COLUMNAS_ENTRADA], y_test=prueba[columna_codigo].astype(int),
        pliegues=pliegues_target,
    )

# Verificación de cada pliegue: periodos disjuntos entre ajuste y validación, y las tres clases en validación
filas_pliegues = []
for target, datos in datos_por_target.items():
    for numero, (ajuste, validacion) in enumerate(datos.pliegues):
        periodos_ajuste = set(datos.periodo_train.iloc[ajuste])
        periodos_validacion = set(datos.periodo_train.iloc[validacion])
        assert periodos_ajuste.isdisjoint(periodos_validacion), f"{target}: periodos compartidos en el pliegue {numero}"
        assert len(ajuste) + len(validacion) == len(datos.y_train)
        clases_validacion = datos.y_train.iloc[validacion]
        assert clases_validacion.nunique() == len(CLASES), f"{target}: falta una clase en el pliegue {numero}"
        filas_pliegues.append({
            "Target": target, "Repetición": numero // N_PLIEGUES, "Pliegue": numero % N_PLIEGUES,
            "Registros de validación": len(validacion), "Periodos de validación": len(periodos_validacion),
            "% clase baja (validación)": 100 * (clases_validacion == 0).mean(),
        })

tabla_pliegues = pd.DataFrame(filas_pliegues).set_index(["Target", "Repetición", "Pliegue"])
display(tabla_pliegues.round(1).unstack("Target"))
display(tabla_pliegues.groupby("Target").agg(["min", "max"]).round(1))
for target, datos in datos_por_target.items():
    print(f"{target}: entrenamiento {len(datos.y_train)} | prueba {len(datos.y_test)} | pliegues {len(datos.pliegues)}")

Registros de validación          Periodos de validación  \
Target                                 TCH sacarosa                    TCH   
Repetición Pliegue                                                           
0          0                           269      270                     12   
           1                           332      334                     11   
           2                           369      369                     11   
           3                           396      397                     11   
           4                           367      367                     11   
1          0                           294      295                     12   
           1                           456      457                     11   
           2                           327      328                     11   
           3                           458      458                     11   
           4                           198      199                     11   
2          0                           400      401                     12   
           1                           295      296                     11   
           2                           234      235                     11   
           3                           291      292                     11   
           4                           513      513                     11   

                            % clase baja (validación)           
Target             sacarosa                       TCH sacarosa  
Repetición Pliegue                                              
0          0             12                      24.2     20.4  
           1             11                      24.4     15.3  
           2             11                      22.0     27.9  
           3             11                      21.7     25.9  
           4             11                      21.5     16.9  
1          0             12                      20.7     22.4  
           1             11                      25.2     12.7  
           2             11                      22.6     15.9  
           3             11                      24.2     18.8  
           4             11                      15.7     56.3  
2          0             12                      22.5     14.7  
           1             11                      21.7     18.9  
           2             11                      20.1     37.0  
           3             11                      25.4     26.4  
           4             11                      22.8     18.5

Registros de validación      Periodos de validación      \
                             min  max                    min max   
Target                                                             
TCH                          198  513                     11  12   
sacarosa                     199  513                     11  12   

         % clase baja (validación)        
                               min   max  
Target                                    
TCH                           15.7  25.4  
sacarosa                      12.7  56.3

TCH: entrenamiento 1733 | prueba 450 | pliegues 15
sacarosa: entrenamiento 1737 | prueba 450 | pliegues 15


#### Conclusión de 3.4

Los 15 pliegues cumplen todas las verificaciones: en cada repetición, cada periodo queda completo en un solo pliegue; ningún periodo aparece a la vez en ajuste y validación; las tres asignaciones son distintas, y todo pliegue de validación tiene las tres clases en los dos targets. `TCH` usa 1733 registros de entrenamiento y sacarosa 1737, con los mismos 15 pliegues.

| | `TCH` | Sacarosa |
|---|---|---|
| Periodos por pliegue de validación | 11 a 12 | 11 a 12 |
| Registros por pliegue de validación | 198 a 513 | 199 a 513 |
| % de clase baja en validación | 15.7 a 25.4 | 12.7 a 56.3 |

- Todos los pliegues de validación tienen 11 o 12 periodos, pero su número de registros va de 198 a 513, porque los periodos tienen tamaños muy distintos: algunos meses concentran muchas cosechas.
- En `TCH`, la proporción de la clase baja es estable entre pliegues (15.7 % a 25.4 %, frente a 22.6 % en todo el entrenamiento), coherente con que el tonelaje no tiene estacionalidad.
- En sacarosa, la proporción de la clase baja cambia mucho: va de 12.7 % a 56.3 %, frente a 21.5 % en todo el entrenamiento. La sacarosa depende del mes de cosecha, así que su clase baja se concentra en los periodos de ciertos meses, y la composición de cada pliegue depende de los meses que reúna. El pliegue con 56.3 % (repetición 1, pliegue 4) es además el más pequeño (199 registros).

Esta variación tiene dos consecuencias para la comparación de modelos. Las métricas de un modelo van a cambiar bastante de un pliegue a otro, sobre todo en sacarosa, así que su media sola no basta para distinguir dos modelos. En cambio, todos los modelos se evalúan en los mismos pliegues, con la misma composición de clases, así que la diferencia entre dos modelos en un mismo pliegue sí es comparable. Por eso la regla de elección de la sección 4 se basa en diferencias pliegue a pliegue, y los 15 pliegues reducen el peso de una asignación particular.

### 3.5 Auditoría de fuga de información

Una fuga de información ocurre cuando el modelo usa, durante el ajuste o la evaluación, datos que no estarían disponibles al aplicarlo. El resultado son métricas optimistas que no se reproducen en operación. Antes de ajustar cualquier modelo, se verifican de forma explícita las fuentes posibles de fuga en este problema:

| Riesgo | Verificación |
|---|---|
| Cosechas del mismo periodo en entrenamiento y prueba | Ningún periodo compartido entre los dos conjuntos |
| Cosechas del mismo periodo en ajuste y validación | Ningún periodo compartido en los 15 pliegues de los dos targets |
| Clases construidas con estadísticos del dataset | Las clases se reconstruyen con los umbrales fijos del Ingenio a partir de su propio target |
| Targets, identificadores o variables medidas después del corte entre las columnas de entrada | Ni `TCH`, ni sacarosa, ni sus clases, ni `FAZ`, `TAL`, `suerte` o `periodo` llegan al modelo. `vejez` y `me` no están en el archivo del EDA |
| Variables de las pruebas de robustez en el modelo principal | `pct_diatrea` y `anio` no aparecen en la matriz de diseño del modelo principal |
| Estadísticos del preprocesamiento calculados con datos de validación | Las medianas, el tope de la lluvia y el escalado cambian de un pliegue a otro, porque se ajustan solo con los datos de ajuste de cada pliegue |
| Uso del conjunto de prueba antes de la evaluación final | Todo acceso al conjunto de prueba pasa por la función `datos_prueba`, que falla mientras la evaluación final no esté habilitada. Se habilita solo en la sección 8 |

Además se cuantifica una característica de la partición que no es una fuga, pero que hay que declarar. Al agrupar por periodo, una suerte puede tener cosechas en los dos conjuntos, en periodos distintos. Que el entrenamiento incluya cosechas **anteriores** de una suerte de prueba corresponde al escenario de uso, porque al clasificar un periodo ya se conoce el resultado de los ciclos previos. Pero el entrenamiento también puede incluir cosechas **posteriores** de esa suerte, que no estarían disponibles en operación. Se cuenta cuántas cosechas de prueba están en cada caso.

In [28]:
# 3.5 Auditoría de fuga de información antes de ajustar cualquier modelo
PRUEBA_HABILITADA = False  # se activa solo en la sección 8, para la evaluación final


def datos_prueba(target):
    """Único acceso al conjunto de prueba de un target. Falla mientras la evaluación final no esté habilitada."""
    assert PRUEBA_HABILITADA, "El conjunto de prueba solo se usa en la sección 8"
    datos = datos_por_target[target]
    return datos.X_test, datos.y_test


auditoria = []


def registrar(riesgo, resultado, cumple):
    """Agrega una verificación a la tabla de la auditoría."""
    auditoria.append({"Riesgo": riesgo, "Resultado": resultado, "Cumple": bool(cumple)})


# Periodos compartidos entre entrenamiento y prueba, y entre ajuste y validación en los 15 pliegues de cada target
periodos_compartidos = set(df_train["periodo"]) & set(df_test["periodo"])
registrar("Periodos en entrenamiento y prueba", f"{len(periodos_compartidos)} compartidos",
          len(periodos_compartidos) == 0)
compartidos_pliegues = sum(len(set(datos.periodo_train.iloc[ajuste]) & set(datos.periodo_train.iloc[validacion]))
                           for datos in datos_por_target.values() for ajuste, validacion in datos.pliegues)
registrar("Periodos en ajuste y validación (15 pliegues × 2 targets)", f"{compartidos_pliegues} compartidos",
          compartidos_pliegues == 0)

# Clases: se reconstruyen con los umbrales fijos a partir de su propio target (cálculo de la sección 1)
clases_coinciden = (
    (clase_tch_reconstruida.astype(str) == df_ipsa.loc[validos_tch, "clase_tch_ingenio"].astype(str)).all()
    and (clase_sac_reconstruida.astype(str) == df_ipsa["clase_sac_ingenio"].astype(str)).all()
)
registrar("Clases construidas con estadísticos del dataset", "umbrales fijos del Ingenio", clases_coinciden)

# Targets, identificadores y variables medidas después del corte: no deben estar entre las columnas de entrada
COLUMNAS_PROHIBIDAS = ["TCH", "sacarosa", "excluir_tch", *COLUMNA_CLASE.values(), *COLUMNA_CODIGO.values(),
                       "FAZ", "TAL", "suerte", "periodo", "NOME", "vejez", "me"]
prohibidas_en_entrada = set(COLUMNAS_PROHIBIDAS) & set(COLUMNAS_ENTRADA)
registrar("Targets, identificadores o variables del molino en la entrada",
          ", ".join(sorted(prohibidas_en_entrada)) or "ninguna", not prohibidas_en_entrada)
medidas_en_molino = {"vejez", "me"} & set(df_ipsa.columns)
registrar("vejez y me en el archivo del EDA", ", ".join(sorted(medidas_en_molino)) or "ausentes",
          not medidas_en_molino)

# Las variables de robustez no llegan a la matriz de diseño del modelo principal
datos_sac = datos_por_target["sacarosa"]
columnas_modelo_principal = crear_preprocesador().fit(datos_sac.X_train).get_feature_names_out()
robustez_en_modelo = set(PREDICTORES_ROBUSTEZ) & set(columnas_modelo_principal)
registrar("pct_diatrea y anio en el modelo principal", ", ".join(sorted(robustez_en_modelo)) or "ausentes",
          not robustez_en_modelo)

# Estadísticos del preprocesamiento: se recalculan con los datos de ajuste de cada pliegue (repetición 0, sacarosa)
filas_estadisticos = []
for numero_pliegue, (ajuste, _) in enumerate(datos_sac.pliegues[:N_PLIEGUES]):
    preprocesador_pliegue = crear_preprocesador().fit(datos_sac.X_train.iloc[ajuste])
    imputador_pliegue = preprocesador_pliegue.named_steps["columnas"].named_transformers_["lluvias"]
    escalador_pliegue = preprocesador_pliegue.named_steps["escalado"]
    filas_estadisticos.append({
        "Ajuste": f"Pliegue {numero_pliegue}", "Registros de ajuste": len(ajuste),
        "Mediana general de lluvias (mm)": imputador_pliegue.mediana_global_,
        "Tope de lluvias (mm)": imputador_pliegue.tope_,
        "Media de edad usada en el escalado": escalador_pliegue.mean_[COLUMNAS_DISENO.index("edad")],
    })
filas_estadisticos.append({
    "Ajuste": "Entrenamiento completo", "Registros de ajuste": len(df_train),
    "Mediana general de lluvias (mm)": imputador_train.mediana_global_, "Tope de lluvias (mm)": imputador_train.tope_,
    "Media de edad usada en el escalado":
        preprocesador_train.named_steps["escalado"].mean_[COLUMNAS_DISENO.index("edad")],
})
tabla_estadisticos = pd.DataFrame(filas_estadisticos).set_index("Ajuste")
topes_distintos = tabla_estadisticos["Tope de lluvias (mm)"].nunique()
registrar("Estadísticos del preprocesamiento calculados con datos de validación",
          f"{topes_distintos} topes de lluvia distintos en 6 ajustes", topes_distintos > 1)

# Acceso al conjunto de prueba antes de la sección 8: debe fallar
try:
    datos_prueba("TCH")
    prueba_bloqueada = False
except AssertionError:
    prueba_bloqueada = True
registrar("Acceso al conjunto de prueba antes de la sección 8", "bloqueado" if prueba_bloqueada else "permitido",
          prueba_bloqueada)

tabla_auditoria = pd.DataFrame(auditoria).set_index("Riesgo")
display(tabla_auditoria)
display(tabla_estadisticos.round(2))
assert tabla_auditoria["Cumple"].all(), "La auditoría encontró una posible fuga de información"

# Suertes con cosechas en los dos conjuntos: anteriores (disponibles en operación) y posteriores (no disponibles)
primer_periodo_train = df_train.groupby("suerte")["periodo"].min()
ultimo_periodo_train = df_train.groupby("suerte")["periodo"].max()
filas_suertes = []
for target, columna_codigo in COLUMNA_CODIGO.items():
    prueba_target = df_test[df_test[columna_codigo].notna()]
    suerte_en_train = prueba_target["suerte"].isin(primer_periodo_train.index)
    con_anterior = prueba_target["periodo"] > prueba_target["suerte"].map(primer_periodo_train)
    con_posterior = prueba_target["periodo"] < prueba_target["suerte"].map(ultimo_periodo_train)
    filas_suertes.append({
        "Target": target, "Cosechas de prueba": len(prueba_target),
        "Su suerte tiene cosechas en entrenamiento": int(suerte_en_train.sum()),
        "% del total": 100 * suerte_en_train.mean(),
        "Con alguna cosecha anterior en entrenamiento": int(con_anterior.sum()),
        "Con alguna cosecha posterior en entrenamiento": int(con_posterior.sum()),
        "Su hacienda tiene cosechas en entrenamiento": int(prueba_target["FAZ"].isin(df_train["FAZ"]).sum()),
    })
display(pd.DataFrame(filas_suertes).set_index("Target").round(1))

,Resultado,Cumple
Riesgo,,
Periodos en entrenamiento y prueba,0 compartidos,True
Periodos en ajuste y validación (15 pliegues × 2 targets),0 compartidos,True
Clases construidas con estadísticos del dataset,umbrales fijos del Ingenio,True
"Targets, identificadores o variables del molino en la entrada",ninguna,True
vejez y me en el archivo del EDA,ausentes,True
pct_diatrea y anio en el modelo principal,ausentes,True
Estadísticos del preprocesamiento calculados con datos de validación,6 topes de lluvia distintos en 6 ajustes,True
Acceso al conjunto de prueba antes de la sección 8,bloqueado,True


,Registros de ajuste,Mediana general de lluvias (mm),Tope de lluvias (mm),Media de edad usada en el escalado
Ajuste,,,,
Pliegue 0,1467,131.0,770.63,12.79
Pliegue 1,1403,138.5,775.03,12.82
Pliegue 2,1368,137.0,780.00,12.80
Pliegue 3,1340,127.0,685.08,12.80
Pliegue 4,1370,132.0,718.35,12.77
Entrenamiento completo,1737,132.5,748.15,12.80


,Cosechas de prueba,Su suerte tiene cosechas en entrenamiento,% del total,Con alguna cosecha anterior en entrenamiento,Con alguna cosecha posterior en entrenamiento,Su hacienda tiene cosechas en entrenamiento
Target,,,,,,
TCH,450,326,72.4,204,211,414
sacarosa,450,326,72.4,204,211,414


#### Conclusión de 3.5

Las ocho verificaciones se cumplen:

| Riesgo | Resultado |
|---|---|
| Periodos en entrenamiento y prueba | Ninguno compartido |
| Periodos en ajuste y validación (15 pliegues × 2 targets) | Ninguno compartido |
| Clases construidas con estadísticos del dataset | No: umbrales fijos del Ingenio |
| Targets, identificadores o variables del molino en la entrada | Ninguno |
| `vejez` y `me` en el archivo del EDA | Ausentes |
| `pct_diatrea` y `anio` en el modelo principal | Ausentes |
| Estadísticos del preprocesamiento calculados con datos de validación | No: cambian en cada ajuste |
| Acceso al conjunto de prueba antes de la sección 8 | Bloqueado |

Los estadísticos del preprocesamiento confirman que cada pliegue se ajusta solo con sus datos. En los cinco pliegues de la primera repetición, la mediana general de la lluvia va de 127.0 a 138.5 mm y el tope de 685.1 a 780.0 mm, frente a 132.5 y 748.1 mm con todo el entrenamiento. La media de la edad usada en el escalado cambia menos (12.77 a 12.82 meses), porque la edad varía poco entre periodos.

**Suertes con cosechas en los dos conjuntos.** Las cifras son iguales en los dos targets, porque el conjunto de prueba no tiene registros sin clase:

| | Cosechas de prueba | % |
|---|---|---|
| Su suerte tiene cosechas en entrenamiento | 326 de 450 | 72.4 % |
| Con alguna cosecha anterior de su suerte en entrenamiento | 204 | 45.3 % |
| Con alguna cosecha posterior de su suerte en entrenamiento | 211 | 46.9 % |
| Su hacienda tiene cosechas en entrenamiento | 414 | 92.0 % |

- Que el 92.0 % de las cosechas de prueba sea de haciendas conocidas, y el 72.4 % de suertes conocidas, corresponde al escenario de uso: el Ingenio clasifica nuevos periodos de suertes que ya ha cosechado.
- Las 204 cosechas con una cosecha anterior de su suerte en entrenamiento también corresponden a ese escenario, porque el resultado de los ciclos previos ya se conoce al clasificar.
- Las 211 cosechas con una cosecha posterior de su suerte en entrenamiento no lo hacen: el modelo se ajusta con información de ciclos que, en operación, todavía no habrían ocurrido. No es una fuga directa, porque ningún predictor identifica la suerte ni contiene su resultado, pero una cosecha posterior de la misma suerte puede parecerse a la de prueba en sus predictores y en su clase. Evitarlo exigiría una partición temporal, entrenando con los primeros años y probando con los últimos, que dejaría el conjunto de prueba concentrado en unos pocos años y meses. Se mantiene la partición por periodo, y esta característica se declara como limitación del desempeño medido en prueba.

### Conclusión de la sección 3

Las cosechas de una misma hacienda en el mismo periodo son casi duplicadas en sus predictores, y la sacarosa se parece entre haciendas dentro de un mismo mes. Por eso la partición y la validación cruzada se agrupan por periodo: el conjunto de prueba está formado por 16 meses completos (450 cosechas, 20.6 %), sin cosechas simultáneas en entrenamiento, y conserva la proporción de clases con una diferencia máxima de 0.3 puntos porcentuales. La partición reproduce la de AA I. La evaluación final no cubre bien febrero, mayo, octubre ni diciembre, que quedan con menos de 10 cosechas en prueba.

El preprocesador, ajustado con el entrenamiento, no deja faltantes, imputa la lluvia con la mediana de cada mes y no tiene problemas de multicolinealidad (VIF máximo de 2.82). Además, reproduce el F1 macro de validación cruzada de los modelos de AA I, así que aplica las mismas reglas.

Los modelos se comparan en 15 pliegues fijos (3 repeticiones × 5 pliegues), agrupados por periodo y comunes a los dos targets. La proporción de la clase baja de sacarosa varía mucho entre pliegues, de 12.7 % a 56.3 %, así que las comparaciones se hacen pliegue a pliegue.

La auditoría no encontró fugas de información, y el conjunto de prueba queda bloqueado hasta la sección 8. La única característica a declarar es que el entrenamiento incluye cosechas posteriores de 211 de las 450 cosechas de prueba, en otros periodos de la misma suerte.

## 4. Métricas y regla de elección del modelo

Las métricas y la regla para elegir el mejor modelo se fijan antes de ajustar cualquier modelo, para que no se acomoden a los resultados.

### 4.1 Métricas

Sea $n_{ij}$ el número de cosechas de la clase real $i$ clasificadas en la clase $j$ (la matriz de confusión), con las clases ordenadas como bajo = 0, medio = 1 y alto = 2, y $n$ el total de cosechas.

**Métricas sobre la clase asignada**

- **Accuracy** (exactitud): proporción de cosechas bien clasificadas, $\sum_i n_{ii} / n$. Con clases desbalanceadas puede ser alta aunque el modelo nunca prediga la clase pequeña, así que no se usa para decidir.
- **Precision** (precisión) de la clase $c$: proporción de las cosechas clasificadas como $c$ que realmente son $c$, $n_{cc} / \sum_i n_{ic}$.
- **Recall** (sensibilidad) de la clase $c$: proporción de las cosechas de la clase $c$ que el modelo identifica, $n_{cc} / \sum_j n_{cj}$.
- **F1-score** de la clase $c$: media armónica de precision y recall, $F_1 = 2PR/(P + R)$. Solo es alto si las dos lo son.
- **Promedio macro:** media simple de una métrica sobre las tres clases. Da el mismo peso a cada clase, sin importar su tamaño.
- **Precision y recall de la clase baja:** las cosechas de desempeño bajo son las que el Ingenio necesita detectar para intervenir. El recall mide qué proporción de ellas detecta el modelo, y la precision, qué proporción de las cosechas señaladas como bajas lo son realmente. Se leen juntas, porque detectar más cosechas bajas suele implicar más falsas alarmas.
- **Kappa de Cohen:** acuerdo entre las clases predichas y las reales, descontando el que se esperaría por azar:

$$\kappa = \frac{p_o - p_e}{1 - p_e}, \qquad p_o = \frac{1}{n}\sum_i n_{ii}, \qquad p_e = \frac{1}{n^2}\sum_i \Big(\sum_j n_{ij}\Big)\Big(\sum_j n_{ji}\Big).$$

  Vale 0 para un clasificador que no supera al azar y 1 para uno perfecto.
- **Kappa ponderado lineal:** el kappa trata todos los errores igual, pero las clases están ordenadas, y clasificar como alta una cosecha baja es peor que confundir clases vecinas. El kappa ponderado asigna a cada error un peso proporcional a la distancia entre las clases, $w_{ij} = |i - j| / 2$:

$$\kappa_w = 1 - \frac{\sum_{i,j} w_{ij}\, n_{ij}}{\sum_{i,j} w_{ij}\, e_{ij}}, \qquad e_{ij} = \frac{1}{n}\Big(\sum_k n_{ik}\Big)\Big(\sum_k n_{kj}\Big).$$

  Se usan pesos lineales y no cuadráticos porque, con tres clases, los cuadráticos ($0$, $1/4$ y $1$) casi no castigan los errores entre clases vecinas.
- **Tasa de errores extremos:** proporción de cosechas bajas clasificadas como altas o altas clasificadas como bajas, $(n_{02} + n_{20}) / n$. Es el error más costoso para el Ingenio: una cosecha baja clasificada como alta queda sin intervenir.

**Métrica sobre el puntaje de la clase baja**

- **AUC de la clase baja:** área bajo la curva ROC de la clase baja frente a las otras dos. Es la probabilidad de que una cosecha baja reciba un puntaje de riesgo mayor que una cosecha que no lo es. Vale 0.5 para un puntaje sin información y 1 para uno que ordena perfectamente. Mide si el modelo ordena las cosechas según su riesgo, independientemente del umbral de decisión, y es la base para fijar un umbral de alerta en la sección 8. El puntaje es la probabilidad estimada de la clase baja. En SVM es el valor de la función de decisión de esa clase, porque las probabilidades de `SVC` se calibran con una validación cruzada interna aleatoria, que no respeta la agrupación por periodo.

**Sobreajuste.** En cada pliegue se calcula también el F1 macro sobre los datos de ajuste. Una diferencia grande entre el F1 de ajuste y el de validación indica que el modelo memoriza el entrenamiento.

La matriz de confusión se presenta en la evaluación final (sección 8), en conteos y normalizada por la clase real.

**Ejemplo de los dos kappa.** Con seis cosechas, dos de cada clase, dos modelos aciertan cinco de seis. El primero confunde una cosecha baja con una media (error entre clases vecinas) y el segundo confunde una cosecha baja con una alta (error extremo). Los dos tienen la misma accuracy (0.833) y el mismo kappa (0.750), pero el kappa ponderado los distingue: 0.800 para el primero y 0.625 para el segundo. La celda siguiente verifica este ejemplo.

[3] scikit-learn developers, “StratifiedGroupKFold,” scikit-learn 1.9 documentation. [En línea]. Disponible: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html